# TrustFed-IoT — Benchmark Experiments (3 Laptops)

**Before running:** Runtime → Change runtime type → **T4 GPU**

| Laptop | Shard | Command |
|--------|-------|---------|
| Laptop 1 | `--shard-index 0` | Run this notebook with LAPTOP=0 |
| Laptop 2 | `--shard-index 1` | Run this notebook with LAPTOP=1 |
| Laptop 3 | `--shard-index 2` | Run this notebook with LAPTOP=2 |

In [ ]:
#@title Step 1: Install dependencies
!pip install torch torchvision optuna requests matplotlib numpy tqdm

In [ ]:
#@title Step 2: Clone project from GitHub
#@markdown Replace YOUR_USERNAME with your GitHub username
import os

REPO_URL = "https://github.com/YOUR_USERNAME/trustfed-iot.git"  #@param {type:"string"}

!git clone $REPO_URL
PROJECT_DIR = REPO_URL.split("/")[-1].replace(".git", "")
os.chdir(PROJECT_DIR)
print(f"Switched to: {os.getcwd()}")

In [ ]:
#@title Step 3: Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU! Go to Runtime → Change runtime type → GPU")

In [ ]:
#@title Step 4: Rebuild partition cache
!python -m data.build_partition_cache

In [ ]:
#@title Step 5: Run benchmark — choose your laptop number!
#@markdown Set LAPTOP_NUMBER to 0, 1, or 2:
LAPTOP_NUMBER = 0  #@param [0, 1, 2]

!python -m experiments.run_all_experiments \
  --rounds 100 \
  --seeds 1 2 3 4 5 \
  --attacks clean gaussian sign_flip scaling label_flip \
  --methods proposed fedavg multikrum \
  --num-shards 3 --shard-index $LAPTOP_NUMBER \
  --export-zip

In [ ]:
#@title Step 6: Download YOUR shard result
#@markdown After experiments finish, download this ZIP
!zip -r benchmark_shard_${LAPTOP_NUMBER}_of_3.zip \
  results/exports/benchmark_shard_${LAPTOP_NUMBER}_of_3.zip

from google.colab.files import download
download(f'benchmark_shard_{LAPTOP_NUMBER}_of_3.zip')

---
## After ALL 3 laptops finish:

1. Download all 3 ZIP files
2. Copy them to ONE laptop
3. Run the merge cell below

In [ ]:
#@title Step 7: Merge all shards and generate plots (run on ONE laptop only)
#@markdown Upload all 3 ZIP files first, then run this
!python -m experiments.plot_experiments \
  --root results/benchmark/ \
  --import-zip results/exports/benchmark_shard_0_of_3.zip \
  --import-zip results/exports/benchmark_shard_1_of_3.zip \
  --import-zip results/exports/benchmark_shard_2_of_3.zip

In [ ]:
#@title Step 8: Download final plots and summary
!zip -r final_results.zip \
  results/benchmark/figures/ \
  results/benchmark/summary/

from google.colab.files import download
download('final_results.zip')